### Respuestas - Trabajo en clase: Q-Learning con FrozenLake

**Actividad 1 — La Q-table**
* **¿Cuántas filas debe tener?** 16 filas (una por cada estado posible del grid 4x4).
* **¿Cuántas columnas?** 4 columnas (una por cada acción posible).
* **¿Qué representa $Q[s,a]$?** El valor esperado (retorno acumulado a futuro) de tomar la acción $a$ en el estado $s$ y seguir una política óptima a partir de ahí.

**Actividad 2 — Inicio del aprendizaje**
Que los valores sean cero no significa que las acciones sean malas, sino que el agente inicia "en blanco" (sin conocimiento previo). Cuando varias acciones tienen el mismo valor máximo (como al principio), el agente debe desempatar de forma aleatoria para fomentar la exploración equitativa.

**Actividad 3 — Identificar la experiencia**
* **Experiencia:** $(s_t, a_t, r_{t+1}, s_{t+1})$
* En Value Iteration conocíamos todo el modelo matemático. Aquí, no disponemos directamente del **estado siguiente resultante empíricamente ($s_{t+1}$)** ni de la **recompensa observada empíricamente ($r_{t+1}$)** hasta no interactuar físicamente con el ambiente.

**Actividad 4 — Leer una fila de Q**
Si el estado es la celda de inicio y $Q(s, \text{Down})$ tiene el número más alto, la política greedy $\arg\max_a Q(s,a)$ seleccionará la acción "Down" porque es la que promete la mayor recompensa acumulada a largo plazo.

**Actividad 5 — Exploration vs. exploitation**
Durante la evaluación no exploramos porque nuestro objetivo ya no es seguir descubriendo el ambiente ni cometer errores intencionales para aprender. El objetivo es **explotar al 100%** el conocimiento que ya se consolidó en la Q-table para obtener la máxima recompensa posible.

**Actividad 6 — Determinístico vs. estocástico**
* **Ambiente:** Cambia la certidumbre. En estocástico (resbaloso), la acción elegida no siempre resulta en el movimiento deseado.
* **Algoritmo:** La ecuación de Q-Learning **no cambia en lo absoluto**. Al ser *model-free*, el algoritmo promedia las transiciones estocásticas automáticamente a lo largo de los episodios.

**Cierre de clase**
1. **¿Qué almacena $Q(s,a)$?** El retorno total esperado de realizar $a$ en $s$.
2. **¿De dónde sale $\max_{a'}Q(s',a')$?** Es el bootstrapping; usamos nuestra propia estimación del mejor valor posible en el estado siguiente ($s'$) para actualizar el estado actual.
3. **¿Por qué necesitamos $\epsilon$-greedy?** Para obligar al agente a visitar todos los estados y probar acciones que inicialmente parecen malas.
4. **¿Por qué Q-Learning es model-free?** Porque no necesita conocer las reglas internas del juego para aprender; aprende puramente mediante la experiencia de ensayo y error.

In [ ]:
import random
import numpy as np

# ACTIVIDAD 1
def choose_action(Q, state, epsilon, env):
    # 1. Exploración
    if random.random() < epsilon:
        return env.action_space.sample()

    # 2. Explotación (con desempate aleatorio)
    max_q = np.max(Q[state])
    best_actions = np.flatnonzero(Q[state] == max_q)
    return int(np.random.choice(best_actions))

# ACTIVIDAD 2
def update_q(Q, state, action, reward, next_state, alpha, gamma):
    best_next_q = np.max(Q[next_state])
    td_target = reward + gamma * best_next_q
    td_error = td_target - Q[state, action]
    Q[state, action] += alpha * td_error

# ACTIVIDAD 3
def train_q_learning(
    env, Q, episodes=5000, alpha=0.1, gamma=0.95, epsilon=0.1, max_steps=200
):
    rewards = []
    for episode in range(episodes):
        state, _ = env.reset()
        total_reward = 0

        for _ in range(max_steps):
            # 1: escoger acción
            action = choose_action(Q, state, epsilon, env)

            # 2: ejecutar acción en el ambiente
            next_state, reward, terminated, truncated, _ = env.step(action)

            # 3: actualizar Q
            update_q(Q, state, action, reward, next_state, alpha, gamma)

            # 4: actualizar estado y recompensa
            state = next_state
            total_reward += reward

            # 5: terminar si corresponde
            if terminated or truncated:
                break

        rewards.append(total_reward)
    return Q, rewards

### Respuestas - Trabajo en casa: Taxi-v4

**Pregunta 1**
Tiene 500 estados y 6 acciones. Tiene muchos más estados que FrozenLake porque codifica múltiples variables simultáneamente: 25 posiciones del taxi × 5 ubicaciones posibles del pasajero × 4 destinos posibles = 500 combinaciones exactas.

**Pregunta 2**
* $s_t$: El estado codificado actual del taxi antes de moverse.
* $a_t$: La acción aleatoria elegida (ej. moverse al norte).
* $r_{t+1}$: La recompensa recibida (usualmente -1 por paso, o -10 por un error).
* $s_{t+1}$: El nuevo estado en el que quedó el taxi tras la acción.

**Pregunta 3**
$500 \text{ estados} \times 6 \text{ acciones} = 3000$ valores en total dentro de la Q-table.

**Pregunta 4**
* La recompensa inicia en números muy negativos (deambula y recibe penalizaciones constantes de -10 por dropoffs erróneos) y sube drásticamente hacia números positivos (cerca de 8).
* Se estabiliza aproximadamente entre los 1500 y 2000 episodios.
* Muestra una política muy buena pero no una convergencia "perfecta" a la máxima recompensa teórica en cada gráfica, dado que se usó un $\epsilon = 0.1$ fijo, obligando al agente a seguir cometiendo errores aleatorios el 10% del tiempo.

**Pregunta 5**
* El taxi sin entrenamiento se mueve erráticamente, choca con paredes e intenta recoger/dejar pasajeros en casillas vacías (penalización). Rara vez logra la tarea dentro del límite de pasos.
* El agente entrenado traza la ruta más directa y corta, recoge al pasajero, y va directo al destino sin dar pasos innecesarios. Se ve claramente que aprendió por la limpieza y eficiencia de sus movimientos.

**Pregunta 6**
1. La acción con mayor valor Q es aquella que representa el paso óptimo a seguir en ese estado particular.
2. Que una acción tenga un valor Q mayor significa que su "promesa" de recompensa acumulada descontada a largo plazo es matemáticamente superior al resto de opciones.
3. Porque el valor $Q(s,a)$ encapsula tanto el $r_{t+1}$ inmediato como el $\gamma \max Q(s_{t+1},a)$ futuro. Es una evaluación del impacto a futuro completo, no solo del beneficio cortoplacista.

**Pregunta 7 (Experimento modificando $\alpha$)**
Si aumentamos $\alpha$ (ej. a 0.8), el agente le da mucho peso a las experiencias más recientes, lo que hace que la velocidad de aprendizaje (subida de la curva) sea más rápida inicialmente. Sin embargo, pierde estabilidad y la curva de recompensa final oscila fuertemente en lugar de mantenerse plana, ya que el agente "olvida" el promedio histórico fácilmente ante cualquier pequeña variación estocástica.